In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import email
from email import policy
from email.utils import parseaddr, parsedate_to_datetime
import hashlib
import ast
import re

NAZARIO_PATH = "drive/MyDrive/phishing project datasets/Nazario_5.csv"
EMAIL_TEXT_PATH = "drive/MyDrive/phishing project datasets/email_text.csv"

pd.set_option('display.max_colwidth', 200)

In [ ]:
df_naz_raw = pd.read_csv(NAZARIO_PATH)
df_txt_raw = pd.read_csv(EMAIL_TEXT_PATH)

print(f"Nazario_5.csv:    {len(df_naz_raw):,} rows, columns: {list(df_naz_raw.columns)}")
print(f"email_text.csv:   {len(df_txt_raw):,} rows, columns: {list(df_txt_raw.columns)}")

Nazario_5.csv:    3,065 rows, columns: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
email_text.csv:   53,668 rows, columns: ['label', 'text']


In [ ]:
expected_naz_cols = {'sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls'}
expected_txt_cols = {'label', 'text'}

assert expected_naz_cols.issubset(df_naz_raw.columns), \
    f"Nazario_5.csv missing columns: {expected_naz_cols - set(df_naz_raw.columns)}"
assert expected_txt_cols.issubset(df_txt_raw.columns), \
    f"email_text.csv missing columns: {expected_txt_cols - set(df_txt_raw.columns)}"
assert len(df_naz_raw) > 0 and len(df_txt_raw) > 0, "One of the files loaded empty."

print("PASS: both files loaded with expected columns.")
print()
print("Nazario_5.csv sample row:")
print(df_naz_raw.iloc[0])
print()
print("email_text.csv sample row (text, first 300 chars):")
print(df_txt_raw.iloc[0]['text'][:300])

PASS: both files loaded with expected columns.

Nazario_5.csv sample row:
sender                                                                                                                                                                           "Hu, Sylvia" <Sylvia.Hu@ENRON.com>
receiver    "Acevedo, Felecia" <Felecia.Acevedo@ENRON.com>, "Brown, MeCole" <MeCole.Brown@ENRON.com>, "Cash, Michelle" <Michelle.Cash@ENRON.com>, "Castellano, Bonne" <Bonne.Castellano@ENRON.com>, "Johnson, Ri...
date                                                                                                                                                                                Fri, 29 Jun 2001 08:36:09 -0500
subject                                                                                                                                                                 FW: June 29 -- BNA, Inc. Daily Labor Report
body        User ID:  enrondlr\nPW:        bnaweb22\n\n\n -----Original Messag

In [ ]:
n_before = len(df_naz_raw)

df_naz_dedup = df_naz_raw.drop_duplicates(
    subset=['sender', 'receiver', 'subject', 'body'], keep='first'
).reset_index(drop=True)

n_after = len(df_naz_dedup)
print(f"Rows before dedup: {n_before:,}")
print(f"Rows after dedup:  {n_after:,}")
print(f"Removed: {n_before - n_after:,} ({(n_before - n_after) / n_before:.2%})")

Rows before dedup: 3,065
Rows after dedup:  3,065
Removed: 0 (0.00%)


In [ ]:
dup_count = df_naz_dedup.duplicated(subset=['sender', 'receiver', 'subject', 'body']).sum()
assert dup_count == 0, f"Still {dup_count} duplicates remaining!"
assert len(df_naz_dedup) <= n_before

print("PASS: no exact duplicate (sender, receiver, subject, body) rows remain.")

PASS: no exact duplicate (sender, receiver, subject, body) rows remain.


In [ ]:
def flag_malformed_naz(df):
    df = df.copy()
    reasons = []

    for _, row in df.iterrows():
        row_reasons = []
        if pd.isna(row['sender']) or str(row['sender']).strip() == '':
            row_reasons.append('missing_sender')
        if pd.isna(row['subject']) or str(row['subject']).strip() == '':
            row_reasons.append('missing_subject')
        if pd.isna(row['body']) or str(row['body']).strip() == '':
            row_reasons.append('missing_body')
        if pd.isna(row['label']):
            row_reasons.append('missing_label')
        reasons.append(row_reasons)

    df['malformed_reasons'] = reasons
    df['is_malformed'] = df['malformed_reasons'].apply(lambda r: len(r) > 0)
    return df


df_naz_flagged = flag_malformed_naz(df_naz_dedup)

df_naz_clean = df_naz_flagged[~df_naz_flagged['is_malformed']].reset_index(drop=True)
df_naz_malformed = df_naz_flagged[df_naz_flagged['is_malformed']].reset_index(drop=True)

print(f"Clean rows:     {len(df_naz_clean):,}")
print(f"Malformed rows: {len(df_naz_malformed):,} ({len(df_naz_malformed)/len(df_naz_flagged):.2%})")

from collections import Counter
reason_counts = Counter(r for reasons in df_naz_malformed['malformed_reasons'] for r in reasons)
print("Breakdown:")
for reason, cnt in reason_counts.most_common():
    print(f"  {reason:16s}: {cnt:,}")

Clean rows:     3,012
Malformed rows: 53 (1.73%)
Breakdown:
  missing_subject : 50
  missing_sender  : 2
  missing_body    : 2


In [ ]:
assert (df_naz_clean['sender'].str.strip() != '').all()
assert (df_naz_clean['subject'].str.strip() != '').all()
assert (df_naz_clean['body'].str.strip() != '').all()
assert df_naz_clean['label'].notna().all()
assert df_naz_malformed['malformed_reasons'].apply(len).gt(0).all()
assert len(df_naz_clean) + len(df_naz_malformed) == len(df_naz_flagged)

print("PASS: df_naz_clean has no missing critical fields.")

PASS: df_naz_clean has no missing critical fields.


In [ ]:
def parse_address(addr_str):
    """Return (display_name, email_address, domain) from a sender/receiver field."""
    if pd.isna(addr_str) or str(addr_str).strip() == '':
        return None, None, None
    name, addr = parseaddr(str(addr_str))
    domain = addr.split('@')[-1].lower() if addr and '@' in addr else None
    return (name or None), (addr or None), domain


def parse_date_safe(date_str):
    """Try RFC-2822 parsing first, fall back to pandas' general parser."""
    if pd.isna(date_str) or str(date_str).strip() == '':
        return None
    try:
        return parsedate_to_datetime(str(date_str))
    except Exception:
        parsed = pd.to_datetime(date_str, errors='coerce', utc=True)
        return None if pd.isna(parsed) else parsed


def parse_urls(url_field):
    """urls column may be a Python-list-as-string, comma/semicolon separated, or a single URL."""
    if pd.isna(url_field) or str(url_field).strip() == '':
        return []
    s = str(url_field).strip()
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return [str(u).strip() for u in parsed if str(u).strip()]
    except (ValueError, SyntaxError):
        pass
    for sep in [';', ',', '|']:
        if sep in s:
            return [u.strip() for u in s.split(sep) if u.strip()]
    return [s]


df_naz_clean = df_naz_clean.copy()

df_naz_clean[['sender_name', 'sender_address', 'sender_domain']] = df_naz_clean['sender'].apply(
    lambda x: pd.Series(parse_address(x))
)
df_naz_clean[['receiver_name', 'receiver_address', 'receiver_domain']] = df_naz_clean['receiver'].apply(
    lambda x: pd.Series(parse_address(x))
)
df_naz_clean['date_parsed'] = df_naz_clean['date'].apply(parse_date_safe)
df_naz_clean['url_list'] = df_naz_clean['urls'].apply(parse_urls)
df_naz_clean['url_count'] = df_naz_clean['url_list'].apply(len)

df_naz_clean.head(3)

,sender,receiver,date,subject,body,label,urls,malformed_reasons,is_malformed,sender_name,sender_address,sender_domain,receiver_name,receiver_address,receiver_domain,date_parsed,url_list,url_count
0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>","""Acevedo, Felecia"" <Felecia.Acevedo@ENRON.com>, ""Brown, MeCole"" <MeCole.Brown@ENRON.com>, ""Cash, Michelle"" <Michelle.Cash@ENRON.com>, ""Castellano, Bonne"" <Bonne.Castellano@ENRON.com>, ""Johnson, Ri...","Fri, 29 Jun 2001 08:36:09 -0500","FW: June 29 -- BNA, Inc. Daily Labor Report","User ID: enrondlr\nPW: bnaweb22\n\n\n -----Original Message-----\nFrom: \t""BNA Highlights"" <bhighlig@bna.com>\nSent:\tThursday, June 28, 2001 11:10 PM\nTo:\tBNA Highlights\nSubject:\tJune ...",0,"['http://web.bna.com', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5k3h4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5e3e4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5q2p6_', 'http://pubs.b...",[],False,"Hu, Sylvia",Sylvia.Hu@ENRON.com,enron.com,None,None,None,2001-06-29 08:36:09-05:00,"[http://web.bna.com, http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5k3h4_, http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5e3e4_, http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5q2p6_, http://pubs.bna.com/ip...",41
1,"""Webb, Jay"" <Jay.Webb@ENRON.com>","""Lambie, Chris"" <Chris.Lambie@ENRON.com>","Fri, 29 Jun 2001 09:37:04 -0500",NGX failover plan.,"\nHi Chris, \n\nTonight we are rolling out a new report. Currently, only you and Jon have access to it. You'll select it off the main reports menu. It will say ""NGX Download report"". When you...",0,[],[],False,"Webb, Jay",Jay.Webb@ENRON.com,enron.com,"Lambie, Chris",Chris.Lambie@ENRON.com,enron.com,2001-06-29 09:37:04-05:00,[],0
2,"""Symms, Mark"" <Mark.Symms@ENRON.com>","""Thomas, Paul D."" <Paul.D.Thomas@ENRON.com>","Fri, 29 Jun 2001 08:39:30 -0500",RE: Intranet Site,"Rika r these new?\n\n -----Original Message-----\nFrom: \tThomas, Paul D. \nSent:\tFriday, June 29, 2001 8:39 AM\nTo:\tSymms, Mark\nSubject:\tIntranet Site\n\nMark,\nWe need a few links to added ...",0,"['http://eastpower.dev.corp.enron.com/summary/pjmsummary.asp', 'http://eastpower.dev.corp.enron.com/summary/nyisosummary.asp', 'http://eastpower.dev.corp.enron.com/Cooper/MID_NYISO.ASP']",[],False,"Symms, Mark",Mark.Symms@ENRON.com,enron.com,"Thomas, Paul D.",Paul.D.Thomas@ENRON.com,enron.com,2001-06-29 08:39:30-05:00,"[http://eastpower.dev.corp.enron.com/summary/pjmsummary.asp, http://eastpower.dev.corp.enron.com/summary/nyisosummary.asp, http://eastpower.dev.corp.enron.com/Cooper/MID_NYISO.ASP]",3


In [ ]:
new_cols = ['sender_name', 'sender_address', 'sender_domain',
            'receiver_name', 'receiver_address', 'receiver_domain',
            'date_parsed', 'url_list', 'url_count']
for c in new_cols:
    assert c in df_naz_clean.columns, f"Missing column: {c}"

addr_extract_rate = df_naz_clean['sender_address'].notna().mean()
print(f"Sender address extraction rate: {addr_extract_rate:.2%}")
assert addr_extract_rate > 0.8, "Unexpectedly low sender-address extraction rate — inspect raw 'sender' values."

date_parse_rate = df_naz_clean['date_parsed'].notna().mean()
print(f"Date parse rate: {date_parse_rate:.2%}")

print()
print("Sample:")
print(df_naz_clean[['sender', 'sender_address', 'sender_domain', 'date', 'date_parsed', 'url_count']].iloc[0])
print()
print("PASS: metadata columns populated.")

Sender address extraction rate: 99.20%
Date parse rate: 99.97%

Sample:
sender            "Hu, Sylvia" <Sylvia.Hu@ENRON.com>
sender_address                   Sylvia.Hu@ENRON.com
sender_domain                              enron.com
date                 Fri, 29 Jun 2001 08:36:09 -0500
date_parsed                2001-06-29 08:36:09-05:00
url_count                                         41
Name: 0, dtype: object

PASS: metadata columns populated.


In [ ]:
n_before_txt = len(df_txt_raw)

df_txt_dedup = df_txt_raw.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)

n_after_txt = len(df_txt_dedup)
print(f"Rows before dedup: {n_before_txt:,}")
print(f"Rows after dedup:  {n_after_txt:,}")
print(f"Removed: {n_before_txt - n_after_txt:,} ({(n_before_txt - n_after_txt)/n_before_txt:.2%})")

Rows before dedup: 53,668
Rows after dedup:  53,668
Removed: 0 (0.00%)


In [ ]:
dup_count_txt = df_txt_dedup.duplicated(subset=['text']).sum()
assert dup_count_txt == 0, f"Still {dup_count_txt} duplicate texts remaining!"
print("PASS: no exact duplicate 'text' rows remain.")

PASS: no exact duplicate 'text' rows remain.


In [ ]:
def flag_malformed_txt(df):
    df = df.copy()
    reasons = []
    for _, row in df.iterrows():
        row_reasons = []
        if pd.isna(row['text']) or str(row['text']).strip() == '':
            row_reasons.append('missing_text')
        if pd.isna(row['label']):
            row_reasons.append('missing_label')
        reasons.append(row_reasons)
    df['malformed_reasons'] = reasons
    df['is_malformed'] = df['malformed_reasons'].apply(lambda r: len(r) > 0)
    return df


df_txt_flagged = flag_malformed_txt(df_txt_dedup)

df_txt_clean = df_txt_flagged[~df_txt_flagged['is_malformed']].reset_index(drop=True)
df_txt_malformed = df_txt_flagged[df_txt_flagged['is_malformed']].reset_index(drop=True)

print(f"Clean rows:     {len(df_txt_clean):,}")
print(f"Malformed rows: {len(df_txt_malformed):,} ({len(df_txt_malformed)/len(df_txt_flagged):.2%})")

Clean rows:     53,668
Malformed rows: 0 (0.00%)


In [ ]:
assert (df_txt_clean['text'].str.strip() != '').all()
assert df_txt_clean['label'].notna().all()
assert len(df_txt_clean) + len(df_txt_malformed) == len(df_txt_flagged)
print("PASS: df_txt_clean has no missing text or label.")

PASS: df_txt_clean has no missing text or label.


In [ ]:
def parse_text_as_email(raw_text):
    """Attempt full RFC-822 parse. If the text has no recognizable headers,
    fall back to treating the whole string as the body."""
    result = {
        'message_id': None, 'date': None, 'from': None, 'to': None, 'subject': None,
        'content_type': None, 'body': None, 'parse_mode': None,
    }

    msg = email.message_from_string(raw_text, policy=policy.compat32)
    has_headers = any([msg.get('From'), msg.get('Subject'), msg.get('Date'), msg.get('To')])

    if has_headers:
        result['message_id'] = msg.get('Message-ID')
        result['date'] = msg.get('Date')
        result['from'] = msg.get('From')
        result['to'] = msg.get('To')
        result['subject'] = msg.get('Subject')
        result['content_type'] = msg.get('Content-Type')

        body = ''
        try:
            if msg.is_multipart():
                parts = []
                for part in msg.walk():
                    if part.get_content_type() == 'text/plain' and not part.is_multipart():
                        payload = part.get_payload(decode=True)
                        if payload:
                            charset = part.get_content_charset() or 'utf-8'
                            parts.append(payload.decode(charset, errors='replace'))
                body = '\n'.join(parts)
            else:
                payload = msg.get_payload(decode=True)
                if payload is not None:
                    charset = msg.get_content_charset() or 'utf-8'
                    body = payload.decode(charset, errors='replace')
                else:
                    fallback = msg.get_payload()
                    body = fallback if isinstance(fallback, str) else ''
        except Exception:
            body = raw_text  # last resort: keep raw text as body

        result['body'] = body.strip()
        result['parse_mode'] = 'headers_extracted'
    else:
        # No headers detected — treat entire string as the body
        result['body'] = raw_text.strip()
        result['parse_mode'] = 'body_only'

    return result


parsed_records = df_txt_clean['text'].apply(parse_text_as_email)
parsed_df = pd.DataFrame(list(parsed_records))
df_txt_clean = pd.concat([df_txt_clean.reset_index(drop=True), parsed_df.reset_index(drop=True)], axis=1)

print(df_txt_clean['parse_mode'].value_counts())
df_txt_clean.head(3)

parse_mode
body_only    53668
Name: count, dtype: int64


,label,text,malformed_reasons,is_malformed,message_id,date,from,to,subject,content_type,body,parse_mode
0,1,do you feel the pressure to perform and not rising to the occasion try v ia gr a your anxiety will be a thing of the past and you will be back to your old self,[],False,None,None,None,None,None,None,do you feel the pressure to perform and not rising to the occasion try v ia gr a your anxiety will be a thing of the past and you will be back to your old self,body_only
1,0,hi i've just updated from the gulus and i check on other mirrors it seems there is a little typo in debian readme file example http gulus usherbrooke ca debian readme ftp ftp fr debian org debian ...,[],False,None,None,None,None,None,None,hi i've just updated from the gulus and i check on other mirrors it seems there is a little typo in debian readme file example http gulus usherbrooke ca debian readme ftp ftp fr debian org debian ...,body_only
2,1,mega authenticv i a g r a discount pricec i a l i s discount pricedo not miss it click here http www moujsjkhchum com,[],False,None,None,None,None,None,None,mega authenticv i a g r a discount pricec i a l i s discount pricedo not miss it click here http www moujsjkhchum com,body_only


In [ ]:
assert 'body' in df_txt_clean.columns and 'parse_mode' in df_txt_clean.columns
assert df_txt_clean['body'].notna().all()
assert set(df_txt_clean['parse_mode'].unique()).issubset({'headers_extracted', 'body_only'})

header_rate = (df_txt_clean['parse_mode'] == 'headers_extracted').mean()
print(f"Rows with extractable headers: {header_rate:.2%}")
print(f"Rows treated as body-only:     {1 - header_rate:.2%}")
print()
print("PASS: every row has a non-null body and a recorded parse_mode.")

Rows with extractable headers: 0.00%
Rows treated as body-only:     100.00%

PASS: every row has a non-null body and a recorded parse_mode.


In [ ]:
def content_hash(text):
    return hashlib.sha256(str(text).encode('utf-8', errors='replace')).hexdigest()

df_txt_clean['content_hash'] = df_txt_clean['body'].apply(content_hash)
n_before_final = len(df_txt_clean)
df_txt_final = df_txt_clean.drop_duplicates(subset=['content_hash'], keep='first').reset_index(drop=True)
n_after_final = len(df_txt_final)

print(f"Rows before final dedup: {n_before_final:,}")
print(f"Rows after final dedup:  {n_after_final:,}")
print(f"Removed as body-duplicates: {n_before_final - n_after_final:,}")

Rows before final dedup: 53,668
Rows after final dedup:  53,661
Removed as body-duplicates: 7


In [ ]:
assert df_txt_final.duplicated(subset=['content_hash']).sum() == 0
print("PASS: no duplicate bodies remain in df_txt_final.")

PASS: no duplicate bodies remain in df_txt_final.


In [ ]:
NAZ_OUT = "drive/MyDrive/phishing project datasets/nazario5_cleaned.csv"
NAZ_MALFORMED_OUT = "drive/MyDrive/phishing project datasets/nazario5_malformed.csv"
TXT_OUT = "drive/MyDrive/phishing project datasets/email_text_cleaned.csv"
TXT_MALFORMED_OUT = "drive/MyDrive/phishing project datasets/email_text_malformed.csv"

df_naz_clean.to_csv(NAZ_OUT, index=False)
df_naz_malformed.to_csv(NAZ_MALFORMED_OUT, index=False)

output_cols = [c for c in df_txt_final.columns if c != 'content_hash']
df_txt_final[output_cols].to_csv(TXT_OUT, index=False)
df_txt_malformed.to_csv(TXT_MALFORMED_OUT, index=False)

print("Saved:")
print(f"  {NAZ_OUT}  ({len(df_naz_clean):,} rows)")
print(f"  {TXT_OUT}  ({len(df_txt_final):,} rows)")

Saved:
  drive/MyDrive/phishing project datasets/nazario5_cleaned.csv  (3,012 rows)
  drive/MyDrive/phishing project datasets/email_text_cleaned.csv  (53,661 rows)
